In [1]:
import pandas as pd
import numpy as np



In [2]:
train_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/train.csv")
X_train = train_data[["id"]]
y_train = train_data[["winner_model_a", "winner_model_b", "winner_tie"]]

test_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/test.csv")
X_test = test_data[["id"]]

print(X_train.head())
print(y_train.head())
print(X_test.head())



           id
0  2444074745
1  1805535695
2  2454781969
3  2915808846
4  3639701142
   winner_model_a  winner_model_b  winner_tie
0               0               1           0
1               1               0           0
2               0               1           0
3               0               1           0
4               1               0           0
           id
0  3297560222
1  2556155375
2  1793939629
3  2562993561
4  2111345399


In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)



In [4]:
# Log loss requires valid probabilities: non-negative and rows summing to 1.
# LinearRegression can output negatives / values not summing to 1, which would yield an invalid or very poor log loss.
# We minimally post-process predictions by clipping and renormalizing (core model/training remains unchanged).
eps = 1e-6
pred = np.clip(predictions, eps, None)
row_sums = pred.sum(axis=1, keepdims=True)
# Safety: if any row somehow sums to 0 after clipping, fall back to uniform probabilities for that row.
row_sums[row_sums == 0] = 3.0
pred = pred / row_sums



In [5]:
# Fix submission schema: required columns are winner_model_a, winner_model_b, winner_tie (not winner_model_c).
submission = pd.DataFrame(
    {
        "id": test_data["id"],
        "winner_model_a": pred[:, 0],
        "winner_model_b": pred[:, 1],
        "winner_tie": pred[:, 2],
    }
)

submission.head()



,id,winner_model_a,winner_model_b,winner_tie
0,3297560222,0.346793,0.340939,0.312267
1,2556155375,0.348462,0.340921,0.310616
2,1793939629,0.350179,0.340903,0.308919
3,2562993561,0.348447,0.340922,0.310631
4,2111345399,0.349464,0.340911,0.309625


In [6]:
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", submission.shape)
print("Columns:", submission.columns.tolist())

Wrote submission.csv with shape: (5748, 4)
Columns: ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
